# Superstore Sales Analytics

This notebook is the exploratory version of the project. I use it to clean the Superstore dataset, understand the sales patterns, and build a first forecasting baseline. One important note: the provided CSV does not include profit, quantity, or discount columns, so I kept the analysis focused on sales, orders, customers, products, and shipping timing.

## 1. Setup

The imports are kept close to what the scripts use. I prefer doing this because it makes it easier to move notebook work into a repeatable pipeline later.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 2. Load the Data

The file is stored inside `data/raw` so the project can run without depending on my Downloads folder.

In [ ]:
DATA_PATH = Path('data/raw/train.csv')
df = pd.read_csv(DATA_PATH, encoding='latin1')
print(df.shape)
df.head()

## 3. Quick Data Audit

Before cleaning, I check data types and missing values. This is where I noticed that postal code has a few blanks, while the main business fields are complete.

In [ ]:
display(df.dtypes)
display(df.isna().sum())
display(df.describe(include='all'))

## 4. Cleaning

The main cleaning work is date parsing and postal-code handling. I fill missing postal codes with `0` instead of dropping rows because those orders still have city, state, and region information.

In [ ]:
cleaned = df.copy()
cleaned['Order Date'] = pd.to_datetime(cleaned['Order Date'], format='%d/%m/%Y', errors='coerce')
cleaned['Ship Date'] = pd.to_datetime(cleaned['Ship Date'], format='%d/%m/%Y', errors='coerce')
cleaned['Postal Code'] = cleaned['Postal Code'].fillna(0).astype(int).astype(str)
cleaned['Ship Delay Days'] = (cleaned['Ship Date'] - cleaned['Order Date']).dt.days
cleaned = cleaned.dropna(subset=['Order Date', 'Ship Date', 'Sales'])
cleaned = cleaned[cleaned['Sales'] >= 0].reset_index(drop=True)
print(cleaned.shape)
cleaned.head()

## 5. Feature Engineering

These date features make EDA easier and also help if I want to test a machine-learning forecasting model later.

In [ ]:
cleaned['Year'] = cleaned['Order Date'].dt.year
cleaned['Month'] = cleaned['Order Date'].dt.month
cleaned['Month Name'] = cleaned['Order Date'].dt.month_name()
cleaned['Quarter'] = cleaned['Order Date'].dt.quarter
cleaned['Order Month'] = cleaned['Order Date'].dt.to_period('M').dt.to_timestamp()
cleaned[['Order Date', 'Year', 'Month', 'Quarter', 'Order Month', 'Ship Delay Days']].head()

## 6. Sales Over Time

Monthly sales give a cleaner signal than daily sales. Daily revenue is useful for forecasting, but for a stakeholder chart I would start with month-level trends.

In [ ]:
monthly_sales = cleaned.groupby('Order Month', as_index=False)['Sales'].sum()
fig = px.line(monthly_sales, x='Order Month', y='Sales', markers=True, title='Monthly Sales Trend')
fig.show()

## 7. Sales by Region, Category, and Segment

These grouped views are simple, but they answer the questions business users usually ask first: where sales come from, which categories matter, and which customer segments dominate.

In [ ]:
for column in ['Region', 'Category', 'Segment', 'Ship Mode']:
    summary = cleaned.groupby(column, as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
    fig = px.bar(summary, x=column, y='Sales', title=f'Sales by {column}')
    fig.show()

## 8. Top Products

Product-level sales are a good place to look for concentration risk. A few high-ticket products can make the total look stronger than the usual order pattern.

In [ ]:
top_products = cleaned.groupby('Product Name', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False).head(10)
px.bar(top_products, x='Sales', y='Product Name', orientation='h', title='Top 10 Products by Sales').show()

## 9. Daily Sales Table for Forecasting

For the model, I aggregate sales to one row per day and reindex missing days to zero sales. This gives the model a proper daily frequency.

In [ ]:
daily_sales = cleaned.groupby('Order Date', as_index=False).agg(Sales=('Sales', 'sum'), Orders=('Order ID', 'nunique')).sort_values('Order Date')
full_index = pd.date_range(daily_sales['Order Date'].min(), daily_sales['Order Date'].max(), freq='D')
daily_sales = daily_sales.set_index('Order Date').reindex(full_index).rename_axis('Order Date').fillna({'Sales': 0, 'Orders': 0}).reset_index()
daily_sales.head(), daily_sales.tail()

## 10. Train/Test Split

I use the last 30 days as a holdout. It is not perfect, but it is a reasonable first check before spending time on a bigger backtesting setup.

In [ ]:
series = daily_sales.set_index('Order Date')['Sales'].asfreq('D')
test_days = 30
train_series = series.iloc[:-test_days]
test_series = series.iloc[-test_days:]
print(train_series.index.min(), train_series.index.max())
print(test_series.index.min(), test_series.index.max())

## 11. Forecasting Model

I chose SARIMAX with weekly seasonality because retail order data often has weekly rhythm, and the model is easy to explain. I would not call this final without comparing a few alternatives.

In [ ]:
model = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fitted_model = model.fit(disp=False)
forecast = fitted_model.forecast(steps=len(test_series))
forecast.head()

## 12. Evaluation

MAE is the metric I would lead with because it reads as average dollar error. RMSE is still useful because it punishes bigger misses.

In [ ]:
mae = mean_absolute_error(test_series, forecast)
rmse = np.sqrt(mean_squared_error(test_series, forecast))
print(f'MAE: {mae:,.2f}')
print(f'RMSE: {rmse:,.2f}')

comparison = pd.DataFrame({'Actual': test_series, 'Forecast': forecast})
px.line(comparison, y=['Actual', 'Forecast'], title='Actual vs Forecast - Holdout Period').show()

## 13. Save Outputs

The dashboard expects the trained model in `models/sales_forecast_sarimax.joblib`. I save the model and metrics together so the app or a later report can show how the model performed.

In [ ]:
Path('models').mkdir(exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
cleaned.to_csv('data/processed/cleaned_superstore.csv', index=False)
daily_sales.to_csv('data/processed/daily_sales.csv', index=False)
joblib.dump({'model': fitted_model, 'metrics': {'mae': float(mae), 'rmse': float(rmse), 'test_days': test_days}}, 'models/sales_forecast_sarimax.joblib')

## 14. Final Notes

The dashboard and script now have enough to support a first analytics review. The biggest limitation is not modeling technique; it is missing business fields like profit and discount. Next time I would ask for those columns and run a more honest profitability analysis alongside the revenue view.